In [2]:
import matplotlib.pyplot as plt
import numpy as np

def plot_image(image, title=''):
    plt.imshow(image)
    plt.axis('off')
    plt.title(title, size=20)

def plot_image2(image, title=''):
    imager = image[...,::-1]
    plt.imshow(imager)
    plt.axis('off')
    plt.title(title, size=20)

In [ ]:
from google.colab.patches import cv2_imshow
import cv2

img=cv2.imread('soccer.jpg')
gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)

grad_x=cv2.Sobel(gray,cv2.CV_32F,1,0,ksize=3) # 형 변환
grad_y=cv2.Sobel(gray,cv2.CV_32F,0,1,ksize=3)

sobel_x=cv2.convertScaleAbs(grad_x)	# 이미지 크기 변환
sobel_y=cv2.convertScaleAbs(grad_y)

edge_strength=cv2.addWeighted(sobel_x,0.5,sobel_y,0.5,0)	# 에지 강도 계산

plt.figure(figsize=(20,10))
plt.gray()
plt.subplot(231), plot_image(gray, 'Source')
plt.subplot(234), plot_image(sobel_x, 'Sobel X')
plt.subplot(235), plot_image(sobel_y, 'Sobel Y')
plt.subplot(236), plot_image(edge_strength, 'Edge Strength')
plt.show()

In [ ]:
# Canny Edge
from google.colab.patches import cv2_imshow
import cv2

def nonmax_suppression(sobel, direct):
    rows, cols = sobel.shape[:2]
    dst = np.zeros((rows, cols), np.float32)
    for i in range(1, rows - 1):
        for j in range(1, cols - 1):
            values = sobel[i-1:i+2, j-1:j+2].flatten()
            first = [3, 0, 1, 2]
            id = first[direct[i, j]]
            v1, v2 = values[id], values[8-id]
            dst[i, j] = sobel[i, j] if (v1 < sobel[i , j] > v2) else 0
    return dst

def trace(max_sobel, i, j, low):
    h, w = max_sobel.shape
    if (0 <= i < h and 0 <= j < w) == False: return  # 추적 화소 범위 확인
    if pos_ck[i, j] == 0 and max_sobel[i, j] > low:
        pos_ck[i, j] = 255
        canny[i, j] = 255
        trace(max_sobel, i - 1, j - 1, low) # 추적 함수 재귀 호출 - 8방향 추적
        trace(max_sobel, i    , j - 1, low)
        trace(max_sobel, i + 1, j - 1, low)
        trace(max_sobel, i - 1, j    , low)
        trace(max_sobel, i + 1, j    , low)
        trace(max_sobel, i - 1, j + 1, low)
        trace(max_sobel, i    , j + 1, low)
        trace(max_sobel, i + 1, j + 1, low)

def hysteresis_th(max_sobel, low, high):                # 이력 임계값 수행
    rows, cols = max_sobel.shape[:2]
    for i in range(1, rows - 1):  # 에지 영상 순회
        for j in range(1, cols - 1):
            if max_sobel[i, j] > high:  trace(max_sobel, i, j, low)  # 추적 시작

image = cv2.imread("canny.jpg", cv2.IMREAD_GRAYSCALE)

pos_ck = np.zeros(image.shape[:2], np.uint8)
canny = np.zeros(image.shape[:2], np.uint8)

# 사용자 정의 캐니 에지
gaus_img = cv2.GaussianBlur(image, (5, 5), 0.3)
Gx = cv2.Sobel(np.float32(gaus_img), cv2.CV_32F, 1, 0, 3)  # x방향 마스크
Gy = cv2.Sobel(np.float32(gaus_img), cv2.CV_32F, 0, 1, 3)  # y방향 마스크
sobel = cv2.magnitude(Gx, Gy)                            # 두 행렬 벡터 크기
directs = cv2.phase(Gx, Gy) / (np.pi / 4)
directs = directs.astype(int) % 4
max_sobel = nonmax_suppression(sobel, directs)   # 비최대치 억제
hysteresis_th(max_sobel, 100, 150)          # 이력 임계값

# OpenCV 캐니 에지
canny2 = cv2.Canny(image, 100, 150)

plt.figure(figsize=(10,10))
plt.gray()
plt.subplot(131), plot_image(image, 'Source')
plt.subplot(132), plot_image(canny, 'User Canny')
plt.subplot(133), plot_image(canny2, 'OpenCV Canny')
plt.show()


In [ ]:
from google.colab.patches import cv2_imshow
import cv2

img=cv2.imread('apples.jpg')
gray=cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

apples=cv2.HoughCircles(gray,cv2.HOUGH_GRADIENT,1,200,param1=150,param2=20,minRadius=50,maxRadius=120)
for i in apples[0]:
    cv2.circle(img,(int(i[0]),int(i[1])),int(i[2]),(255,0,0),2)

cv2_imshow(img)
